In [1]:
import pandas as pd
import numpy as np

In [2]:
df_titatinc = pd.read_excel("../datasets/titanic.xls")
df_titatinc.shape

(1309, 14)

In [3]:
df_titatinc.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [4]:
df_titatinc.describe()

,pclass,survived,age,sibsp,parch,fare,body
count,1309.000000,1309.000000,1046.000000,1309.000000,1309.000000,1308.000000,121.000000
mean,2.294882,0.381971,29.881135,0.498854,0.385027,33.295479,160.809917
std,0.837836,0.486055,14.413500,1.041658,0.865560,51.758668,97.696922
min,1.000000,0.000000,0.166700,0.000000,0.000000,0.000000,1.000000
25%,2.000000,0.000000,21.000000,0.000000,0.000000,7.895800,72.000000
50%,3.000000,0.000000,28.000000,0.000000,0.000000,14.454200,155.000000
75%,3.000000,1.000000,39.000000,1.000000,0.000000,31.275000,256.000000
max,3.000000,1.000000,80.000000,8.000000,9.000000,512.329200,328.000000


In [5]:
df_titatinc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   int64  
 1   survived   1309 non-null   int64  
 2   name       1309 non-null   object 
 3   sex        1309 non-null   object 
 4   age        1046 non-null   float64
 5   sibsp      1309 non-null   int64  
 6   parch      1309 non-null   int64  
 7   ticket     1309 non-null   object 
 8   fare       1308 non-null   float64
 9   cabin      295 non-null    object 
 10  embarked   1307 non-null   object 
 11  boat       486 non-null    object 
 12  body       121 non-null    float64
 13  home.dest  745 non-null    object 
dtypes: float64(3), int64(4), object(7)
memory usage: 143.3+ KB


## Fare Validation

In [6]:
max_fare = df_titatinc['fare'].max()
max_fare

512.3292

In [7]:
min_fare = df_titatinc['fare'].min()
min_fare

0.0

### Option 1:
Remove rows where the "fare" is 0 or NaN

In [8]:
df1 = df_titatinc[(df_titatinc['fare'] != 0) & (df_titatinc['fare'].notna())]
df1.shape

(1291, 14)

### Option 2:
Replace anamolies (zero/nan) with the "fare" column median (or mean if we want)

📌 The Titanic fare distribution is heavily skewed
Fares in 1st class go up to 512, while many 3rd‑class fares are under 10.
This creates a long right tail — a classic skewed distribution.

In skewed data:
Mean gets pulled upward by extreme values

Median stays stable and represents the “typical” fare better

So the median is more robust.


In [9]:
df2 = df_titatinc.copy()
df2.shape

(1309, 14)

In [10]:
print('fare column: number of NA values', df_titatinc['fare'].isna().sum())

fare column: number of NA values 1


In [11]:
print( 'fare column: number of 0 values', (df_titatinc['fare'] == 0).sum() )

fare column: number of 0 values 17


In [12]:
df2['fare'] = df2['fare'].replace(0, np.nan)
df2['fare'] = df2['fare'].fillna(df2.groupby('pclass')['fare'].transform('median')) # TODO: why transform?!
                                                                                    # Must use transform because it returns a Series aligned row‑by‑row with the original DataFrame
# df2['fare'] = df2['fare'].fillna(df_titatinc.groupby('pclass')['fare'].median()) 

df2.shape


(1309, 14)

In [17]:
max_fare = df2['fare'].max()
max_fare

512.3292

In [18]:
min_fare = df2['fare'].min()
min_fare

3.1708

❌ Option NOT recommended — “exclude values below a threshold”
That introduces bias and artificially reshapes the dataset.
Better to use robust statistics instead of arbitrary cutoffs.

### Option 3:
✅ Winsorize (recommended)
This caps extreme values at a percentile.

Example: cap the bottom 1% and top 1% within each pclass.

In [42]:
df3 = df_titatinc.copy()
df3.shape

(1309, 14)

In [43]:
#conda install scipy

In [44]:
# df3['fare'] = df3['fare'].replace(0, np.nan)

# df3['fare_log'] = np.log1p(df3['fare'])

# df3['fare_log'] = df3['fare_log'].fillna(
#     df3.groupby('pclass')['fare_log'].transform('median')
# )

# df3['fare'] = np.expm1(df3['fare_log'])

#########################################################

from scipy.stats.mstats import winsorize

df3['fare_wins'] = df3.groupby('pclass')['fare'].transform(
    lambda x: winsorize(x, limits=[0.01, 0.01])
)

df3['fare'] = df3['fare'].replace(0, np.nan)
df3['fare'] = df3['fare'].fillna(df3.groupby('pclass')['fare_wins'].transform('median'))


In [48]:
def winsorize_group(x):
    # cap bottom 1% and top 1%
    return winsorize(x, limits=[0.01, 0.01])

df3['fare'] = df3.groupby('pclass')['fare'].transform(winsorize_group)


In [49]:
max_fare = df3['fare'].max()
max_fare

512.3292

In [50]:
min_fare = df3['fare'].min()
min_fare

6.4958

In [51]:
fare_ranges = df3.groupby("pclass")["fare"].agg(["min", "max"])
fare_ranges

,min,max
pclass,,
1,25.7417,512.3292
2,10.5000,73.5000
3,6.4958,69.5500


In [21]:
def validate_fare(fare):
    if fare < min_fare or fare > max_fare:
        print("Fair price is illegal! Please pay again")
        # raise Exception("Fair price is illegal! Please pay again")
        return False

    print("Fare is accepted")
    return True

### Debug

In [22]:
validate_fare(150)

Fare is accepted


True

In [23]:
validate_fare(1)

Fair price is illegal! Please pay again


False

In [24]:
validate_fare(600)

Fair price is illegal! Please pay again


False

## Age validation

In [25]:
min_age = 0
max_age = 130

def validate_age(age):
    if age < min_age or age > max_age:
        print("The age that you had entered is illegal! Please enter your age again")
        return False

    print("Age is accepted")
    return True

### Debug

In [26]:
validate_age(150)

The age that you had entered is illegal! Please enter your age again


False

In [27]:
validate_age(-15)

The age that you had entered is illegal! Please enter your age again


False

In [28]:
validate_age(40)

Age is accepted


True

## Sex validation

In [29]:
def validate_sex(sex):
    if sex not in ('male', 'female'):
        print("The sex that you had entered is illegal! Please enter your gender again")
        return False

    print("Sex is accepted")
    return True

### Debug

In [30]:
validate_sex('male')

Sex is accepted


True

In [31]:
validate_sex('female')

Sex is accepted


True

In [27]:
validate_sex('trans')

The sex that you had entered is illegal! Please enter your gender again


False

# User Input

In [52]:

def user_input():
    name = input("Enter name: ").strip() 
    
    # Validate fare
    while True:
        try:
            fare = float(input("Enter fare: "))
            if validate_fare(fare):
                break
        except ValueError:
            print("Fare must be a number. Try again.")
    
    # Validate age
    while True:
        try:
            age = float(input("Enter age: "))
            if validate_age(age):
                break
        except ValueError:
            print("Age must be a number. Try again.")
    
    # Validate sex
    while True:
        sex = input("Enter sex (male/female): ").lower()
        if validate_sex(sex):
            break
    
    print("\n-------------------------------")
    
    print("\nAll inputs are valid!")
    print("Name:", name)
    print("Age:", age)
    print("Fare:", fare)
    print("Sex:", sex)

    user_input_data = {
        "name": name,
        "age": age,
        "fare": fare,
        "sex": sex
    }

    return user_input_data


In [54]:
passenger = user_input()

Enter name:  Muhammad Z
Enter fare:  3


Fair price is illegal! Please pay again


Enter fare:  600


Fair price is illegal! Please pay again


Enter fare:  350


...

Age is accepted


Enter sex (male/female):  else


The sex that you had entered is illegal! Please enter your gender again


Enter sex (male/female):  male


Sex is accepted

-------------------------------

All inputs are valid!
Name: Muhammad Z
Age: 45.0
Fare: 350.0
Sex: male


In [55]:
passenger

{'name': 'Muhammad Z', 'age': 45.0, 'fare': 350.0, 'sex': 'male'}

# Find the corresponding class of the user fare inpute

### Step 1 - Find fare classes ranges (was calculated already up - using df3)

In [61]:
# fare_ranges = df2.groupby("pclass")["fare"].agg(["min", "max"])
# fare_ranges

fare_ranges = df3.groupby("pclass")["fare"].agg(["min", "max"])
fare_ranges

,min,max
pclass,,
1,25.7417,512.3292
2,10.5000,73.5000
3,6.4958,69.5500


### Step 2 - Class detection logic

In [57]:
def detect_class(fare, fare_ranges):
    # Loop from highest class to lowest
    for pclass in sorted(fare_ranges.index, reverse=False):
        min_fare = fare_ranges.loc[pclass, "min"]
        max_fare = fare_ranges.loc[pclass, "max"]

        if min_fare <= fare <= max_fare:
            return pclass

    print("fare is outside of all ranges")
    return None  # Should not happen unless fare is outside all ranges


### Debug

In [58]:
detect_class(3, fare_ranges)

fare is outside of all ranges


In [59]:
detect_class(600, fare_ranges)

fare is outside of all ranges


In [62]:
detect_class(7, fare_ranges)

3

In [63]:
detect_class(11, fare_ranges)

2

In [64]:
detect_class(30, fare_ranges)

1

### Test with the user input fare

In [65]:
passenger['class'] = detect_class(passenger['fare'], fare_ranges)
passenger

{'name': 'Muhammad Z', 'age': 45.0, 'fare': 350.0, 'sex': 'male', 'class': 1}

# Writing the ticket

In [66]:
import random

def generate_unique_ticket():
    """
    existing_tickets: a Python set or list of existing ticket numbers (as strings)
    returns: a new unique 6-digit ticket number as a string
    """
    existing_tickets = df2['ticket']
    existing = set(str(t) for t in existing_tickets)

    while True:
        # create a random 6-digit number, from 000000 to 999999
        num = random.randint(0, 999999)
        ticket_str = f"{num:06d}"  # ensure it has 6 digits with leading zeros
        if ticket_str not in existing:
            return ticket_str


In [67]:
ticket = generate_unique_ticket()
ticket

'410722'

In [68]:
passenger['ticket'] = ticket
passenger

{'name': 'Muhammad Z',
 'age': 45.0,
 'fare': 350.0,
 'sex': 'male',
 'class': 1,
 'ticket': '410722'}

In [69]:
def write_passenger_ticket_to_file(passenger, filename):
    with open('passenger.txt', 'w') as file:
        file.write("---------------------------------------------\n")
        file.write(f"| ticket: {passenger['ticket']}     | \tfare: {passenger['fare']}       \t|\n")
        file.write("---------------------------------------------\n")
        file.write(f"| age: {int(passenger['age'])}            | \tclass: {passenger['class']}        \t|\n")
        file.write("---------------------------------------------\n")
        file.write(f"| sex: {passenger['sex']}          | \tname: {passenger['name']} \t|\n")
        file.write("---------------------------------------------\n")


In [70]:
write_passenger_ticket_to_file(passenger, "passenger.txt")

# Survival chances of the passenger

## Using groupby

In [71]:
MIN_SAMPLES = 100

In [72]:
age_range = [passenger['age']-5, passenger['age']+5]
# age_range = [130-5, 130+5] # for testing
# age_range = [5-5, 5+5] # for testing
age_range

[40.0, 50.0]

In [73]:
samples_count = len(df2[ (df2['age'] >= age_range[0]) & (df2['age'] <= age_range[1]) ])
print(f"Initial range = {age_range}, samples count = {samples_count}")

rng = age_range.copy() ## lists are mutable and assigned by reference
i=0
while samples_count <= MIN_SAMPLES:
    rng[0] -= 5 # move lower bound down 
    rng[1] += 5 # move upper bound up
    samples_count = len(df2[ (df2['age'] >= rng[0]) & (df2['age'] <= rng[1]) ])
    print(f"Iteration {i}: range={rng}, samples count = {samples_count}")
    i += 1      

samples_count

Initial range = [40.0, 50.0], samples count = 150


150

In [74]:
# samples_count = len(df2[(df2['age'] >= age_range[0]) & (df2['age'] <= age_range[1])])
# print(f"Initial range = {age_range}, samples count = {samples_count}")

# rng = age_range.copy()
# i = 0

# min_age = df2['age'].min()
# max_age = df2['age'].max()

# while samples_count <= sample_group_min:

#     # Stop if we hit dataset boundaries
#     if rng[0] <= min_age and rng[1] >= max_age:
#         print("Reached full dataset age range — cannot reach sample_group_min.")
#         break

#     rng[0] = max(min_age, rng[0] - 5)
#     rng[1] = min(max_age, rng[1] + 5)

#     new_count = len(df2[(df2['age'] >= rng[0]) & (df2['age'] <= rng[1])])
#     print(f"Iteration {i}: range={rng}, samples count = {new_count}")

#     # Stop if no improvement (prevents infinite loop)
#     # if new_count == samples_count:
#     #     print("No further increase in sample count — stopping.")
#     #     break

#     samples_count = new_count
#     i += 1

# samples_count


In [48]:
# df2[ (df2['age'] >= 0) & (df2['age'] <= 10) ].groupby(['sex', 'pclass']).sum(numeric_only=True)
# df2[ (df2['age'] >= 0) & (df2['age'] <= 10) ].groupby(['sex', 'pclass']).count()

In [75]:
stats = df3[ (df3['age'] >= rng[0]) & (df3['age'] <= rng[1]) ].groupby(['sex', 'pclass']).mean(numeric_only=True)
stats

survived        age     sibsp     parch        fare  \
sex    pclass                                                        
female 1       0.960000  45.760000  0.480000  0.520000  112.906840   
       2       0.933333  44.666667  0.333333  0.666667   24.716667   
       3       0.250000  44.333333  0.583333  1.750000   20.298958   
male   1       0.311111  45.677778  0.444444  0.066667   60.737593   
       2       0.047619  43.952381  0.333333  0.238095   20.775990   
       3       0.062500  43.250000  0.218750  0.625000   11.719659   

                     body   fare_wins  
sex    pclass                          
female 1              NaN  112.906840  
       2              NaN   24.716667  
       3         7.000000   20.298958  
male   1       150.666667   58.067500  
       2       172.400000   20.775990  
       3       171.857143   11.667837

In [76]:
# Extract survival probability 
try: 
    passenger['survival_chance'] = stats.loc[(passenger['sex'], passenger['class']), 'survived'] 
    print(f"Dear {passenger['name']}, your chances to die on our trip are {passenger['survival_chance']*100:.2f}%.\nEnjoy your trip ☺")
except KeyError: 
    print("No matching group found for this passenger in the sample range.")

Dear Muhammad Z, your chances to die on our trip are 31.11%.
Enjoy your trip ☺


## Using AI

In [78]:
# conda install -c conda-forge google-genai

In [ ]:
import os
from dotenv import load_dotenv
import google.genai as genai

load_dotenv()

In [80]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
# genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

In [81]:

def survival_chance(passenger):
    # model = genai.GenerativeModel("gemini-pro") # deprecated
    # model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""
    You are an AI assistant helping with a Titanic passenger simulation.

    Based on the following passenger details:
    Name: {passenger['name']}
    Age: {passenger['age']}
    Sex: {passenger['sex']}
    Class: {passenger['class']}
    Survival Chance: {passenger['survival_chance']}

    Create a friendly, creative message that includes:
    1. A short personality description of the passenger.
    2. A travel tip for their Titanic journey.
    3. A fun and short, imaginative survival prediction (not too serious).
    """

    # response = model.generate_content(prompt)

    response = client.models.generate_content(
        model="gemini-2.5-flash", contents=prompt,
    )

    return response.text


In [74]:
# Example
# print(survival_chance(passenger))
"""
Hello there, Mr. Muhammad Z!

Welcome aboard for this grand Titanic simulation! We're delighted to have you.

Here's a little something about your journey:

1.  **Personality Snapshot:** As a distinguished gentleman of 45 in First Class, we picture you as someone with a refined taste, a calm demeanor, and a penchant for engaging conversations, perhaps over a fine cigar or a morning coffee.
2.  **Travel Tip:** Make the most of your luxurious surroundings! We suggest enjoying a leisurely read in the opulent smoking room or a refreshing dip in the heated swimming pool. Don't forget to savor every exquisite meal!
3.  **Survival Prediction:** With a 31% survival chance, Mr. Muhammad, we playfully predict you might end up inadvertently becoming the world's most dapper 'luxury rafter,' safely floating on a particularly plush first-class sofa until rescued by a passing fishing trawler convinced you're a lost dignitary!
"""

Hello there, Mr. Muhammad Z!

Welcome aboard for this grand Titanic simulation! We're delighted to have you.

Here's a little something about your journey:

1.  **Personality Snapshot:** As a distinguished gentleman of 45 in First Class, we picture you as someone with a refined taste, a calm demeanor, and a penchant for engaging conversations, perhaps over a fine cigar or a morning coffee.
2.  **Travel Tip:** Make the most of your luxurious surroundings! We suggest enjoying a leisurely read in the opulent smoking room or a refreshing dip in the heated swimming pool. Don't forget to savor every exquisite meal!
3.  **Survival Prediction:** With a 31% survival chance, Mr. Muhammad, we playfully predict you might end up inadvertently becoming the world's most dapper 'luxury rafter,' safely floating on a particularly plush first-class sofa until rescued by a passing fishing trawler convinced you're a lost dignitary!


In [85]:
result = survival_chance(passenger)
print(result)

Here's a message for Muhammad Z!

Ah, Muhammad Z! At 45, we imagine you as a gentleman of quiet sophistication, perhaps a connoisseur of life's finer details and a delightful conversationalist when the mood strikes.

**Your Titanic Travel Tip:** Make sure to savor at least one magnificent meal in the À la Carte Restaurant! It's an unforgettable culinary experience that truly embodies the luxury of your voyage.

**Your Fun Survival Prediction:** With a 31.1% chance, Muhammad, your tale will be quite the adventure! Our crystal ball hints that you might just survive by being mistaken for a crucial member of the ship's emergency chess team, thus securing a spot on a lifeboat. You'll then spend the waiting hours expertly strategizing against a particularly smug seagull, entirely unfazed by the chilly waters!


In [86]:
with open("passenger.txt", "a", encoding="utf-8") as f:
    f.write("\n\n" + result + "\n\n")